In [ ]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import (
    RECALL_TOP_1_PERCENT,
    RECALL_TOP_5_PERCENT,
    make_objective,
)

In [ ]:
FEATURES = "data/sampled.parquet"
LABELS = "data/1L83.1L83:p2rank:3.output.parquet"

RANDOM_SEED = 1000

features = pl.read_parquet(FEATURES)
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

df = features.join(labels, on="catalog_id", how="inner")

In [ ]:
import logging

from e3fp.pipeline import fprints_from_mol
from rdkit import Chem


def generate_e3fp(sdf_str: str, bits: int = 1024):
    logging.basicConfig(level=logging.WARNING)
    logging.getLogger().setLevel(logging.WARNING)
    for logger_name in logging.root.manager.loggerDict:
        logging.getLogger(logger_name).setLevel(logging.WARNING)

    zero_vector = [0] * bits

    if not sdf_str:
        return zero_vector

    try:
        mol = Chem.MolFromMolBlock(sdf_str)
        if mol is None:
            return zero_vector

        if not mol.HasProp("_Name") or not mol.GetProp("_Name"):
            mol.SetProp("_Name", "molecule")

        fps = fprints_from_mol(mol, fprint_params={"bits": bits})

        if not fps:
            return zero_vector

        return fps[0].to_vector(sparse=False).astype(int).tolist()
    except Exception:
        return zero_vector

In [ ]:
sdf_list = df["conformer_sdf"].to_list()

In [ ]:
import concurrent.futures

from tqdm.auto import tqdm

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(
        tqdm(
            executor.map(generate_e3fp, sdf_list, chunksize=50),
            total=len(sdf_list),
            desc="Generating fingerprints",
        )
    )

In [ ]:
df = df.with_columns(pl.Series("e3fp", results, dtype=pl.List(pl.Int64)))

In [ ]:
ALL_COLUMN_NAMES = [
    "catalog_id",
    "affinity_kcal_mol",
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
    "morgan_fingerprint",
    "e3fp",
]

df = df.select(ALL_COLUMN_NAMES)

In [ ]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x_scalars = df.select(FEATURE_NAMES).to_numpy()

x_morgan = np.array(df["morgan_fingerprint"].to_list())
x_e3fp = np.array(df["e3fp"].to_list())

y = df[LABEL_NAME].to_numpy()

In [33]:
# x = np.hstack([x_scalars, x_morgan])
# x = np.hstack([x_scalars, x_e3fp])

x = np.hstack([x_scalars, x_morgan, x_e3fp])

x_ = x[:100000]
y_ = y[:100000]

# PRIMARY_METRIC = RECALL_TOP_1_PERCENT
PRIMARY_METRIC = RECALL_TOP_5_PERCENT

NUM_TRIALS = 50

x_train, x_test, y_train, y_test = train_test_split(
    x_, y_, test_size=0.75, random_state=RANDOM_SEED
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    make_objective(
        x_train,
        y_train,
        5,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

[I 2026-09-09 00:25:36,859] A new study created in memory with name: no-name-8b5012cf-1871-4020-a17c-975b84a030d1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-09 00:25:42,507] Trial 0 finished with value: 0.1312 and parameters: {'num_leaves': 173, 'max_depth': 4, 'learning_rate': 0.22592582730543753, 'n_estimators': 1016, 'min_child_samples': 88, 'subsample': 0.60616634046136, 'colsample_bytree': 0.5203548123845445, 'reg_alpha': 3.7562124870680294e-05, 'reg_lambda': 1.2536888868437446e-06}. Best is trial 0 with value: 0.1312.
[I 2026-09-09 00:25:52,323] Trial 1 finished with value: 0.14240000000000003 and parameters: {'num_leaves': 218, 'max_depth': 5, 'learning_rate': 0.06905371732106186, 'n_estimators': 845, 'min_child_samples': 22, 'subsample': 0.87176970729607, 'colsample_bytree': 0.5347910404849773, 'reg_alpha': 0.9290409121444837, 'reg_lambda': 3.7480000930162687}. Best is trial 1 with value: 0.14240000000000003.
[I 2026-09-09 00:26:09,996] Trial 2 finished with value: 0.1272 and parameters: {'num_leaves': 240, 'max_depth': 7, 'learning_rate': 0.001179752983913515, 'n_estimators': 1966, 'min_child_samples': 37, 'subsample': 

In [34]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params, deterministic=True, force_row_wise=True)
final_model.fit(
    x_train,
    y_train,
    eval_X=x_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l2: 277.973


,num_leaves,93
,max_depth,12
,learning_rate,0.002734465997120872
,min_child_samples,56
,subsample,0.9923933668748933
,colsample_bytree,0.6815503081403322
,reg_alpha,0.008943974542095053
,reg_lambda,0.009167285251408202
,random_state,1000
,deterministic,True
,force_row_wise,True


In [35]:
y_pred = np.asarray(final_model.predict(x_test))

results = {
    "spearman": spearman_corr(y_test, y_pred),
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "enrichment_factor_1_percent": enrichment_factor(y_test, y_pred, 0.01),
    "enrichment_factor_5_percent": enrichment_factor(y_test, y_pred, 0.05),
    "enrichment_factor_10_percent": enrichment_factor(y_test, y_pred, 0.1),
}

results

{'spearman': 0.26097251587258885,
 'top_1_percent': 0.10933333333333334,
 'top_5_percent': 0.20186666666666667,
 'top_10_percent': 0.2829333333333333,
 'enrichment_factor_1_percent': 10.933333333333334,
 'enrichment_factor_5_percent': 4.037333333333333,
 'enrichment_factor_10_percent': 2.829333333333333}

| train size | train metric | trials | fingerprint(s) | spearman | top_1_percent | top_5_percent | top_10_percent | enrichment 1 percent | enrichment 5 percent | enrichment 10 percent |
| - | - | - | - | - | - | - | - | - | - | - |
| 10000 | RECALL_TOP_1_PERCENT | 50 | morgan + e3 | | | | | | | |
| 10000 | RECALL_TOP_5_PERCENT | 50 | morgan + e3 | | | | | | | |
| 25000 | RECALL_TOP_1_PERCENT | 50 | morgan + e3 | | | | | | | |
| 25000 | RECALL_TOP_5_PERCENT | 50 | morgan + e3 | | | | | | | |
| 50000 | RECALL_TOP_1_PERCENT | 50 | morgan + e3 | 0.371 | 0.154 | 0.266 | 0.326 | 15.4 | 5.33 | 3.26 |
| 50000 | RECALL_TOP_5_PERCENT | 50 | morgan + e3 | 0.373 | 0.068 | 0.228 | 0.310 | 6.80 | 4.55 | 3.09 |
| 75000 | RECALL_TOP_1_PERCENT | 50 | morgan + e3 | | | | | | | |
| 75000 | RECALL_TOP_5_PERCENT | 50 | morgan + e3 | 0.261 | 0.109 | 0.202 | 0.283 | 10.93 | 4.04 | 2.83 |

Targets:

Spearman ρ > 0.5

EF@1% > 10-20 is considered reasonably strong for early enrichment in virtual screening
EF@1% > 30-50 is very good, approaching what you'd see from decent docking scores themselves
